# Week 4 — Cost-to-income ratio & burn multiple

HDFC (a licensed bank) and Jupiter (a neobank operating on a partner-bank/NBFC model) don't
share a directly comparable P&L structure, so this notebook does **not** force a single
ratio across both. Instead it reports each entity's own best-disclosed efficiency metric,
then derives a bounded 0–100 *cost efficiency score* so it can feed the Week 4 composite index.

**Input:** `data/processed/cost_efficiency_clean.csv`

**Output:** `data/processed/cost_efficiency_scores.csv`

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"

df = pd.read_csv(PROCESSED / "cost_efficiency_clean.csv")
df

,entity,metric,value,unit,fiscal_year,source,confidence
0,HDFC Bank,Cost-to-Income Ratio,40.20,%,FY2023-24,HDFC Bank Q4FY24 investor presentation,official
1,HDFC Bank,Operating (non-interest) expenses,63386.00,INR crore,FY2023-24,Equitymaster FY24 annual report analysis,official
2,HDFC Bank,Total operating income (interest+other),407995.00,INR crore,FY2023-24,Equitymaster FY24 annual report analysis,official
3,HDFC Bank,Net profit,64062.00,INR crore,FY2023-24,Equitymaster FY24 annual report analysis,official
4,Jupiter (Amica Financial Technologies),Revenue from operations (parent),35.85,INR crore,FY2023-24,Entrackr Nov 2024,official (roc filing via press)
5,Jupiter (Amica Financial Technologies),Consolidated operating revenue (incl. NBFC arm),51.20,INR crore,FY2023-24,Entrackr Nov 2024,official (roc filing via press)
6,Jupiter (Amica Financial Technologies),Net loss (parent),275.94,INR crore,FY2023-24,Entrackr Nov 2024,official (roc filing via press)
7,Jupiter (Amica Financial Technologies),Net loss (consolidated),233.63,INR crore,FY2023-24,Entrackr Nov 2024,official (roc filing via press)
8,Jupiter (Amica Financial Technologies),Employee benefit expense,195.10,INR crore,FY2023-24,Entrackr Nov 2024,official (roc filing via press)
9,Jupiter (Amica Financial Technologies),Other operating expenses (ex-ESOP),330.10,INR crore,FY2023-24,Entrackr Nov 2024,official (roc filing via press)


## HDFC Bank — Cost-to-Income Ratio

Disclosed directly by the bank (Q4FY24 investor presentation): **40.2%**. We don't
recompute it from `Operating expenses / Total operating income` in this dataset because
the disclosed CIR denominator is *net* interest income + other income, while our raw
figure for "Total operating income" is gross interest income + other income — recomputing
from it would silently understate CIR. Use the disclosed figure.

In [2]:
hdfc_cir = df.loc[
    (df["entity"] == "HDFC Bank") & (df["metric"] == "Cost-to-Income Ratio"), "value"
].iloc[0]
print(f"HDFC Bank Cost-to-Income Ratio: {hdfc_cir}%")

# Bounded score: lower CIR is better. 0% CIR -> 100, 100%+ CIR -> 0.
hdfc_cost_efficiency_score = max(0.0, 100.0 - hdfc_cir)
print(f"HDFC Bank cost efficiency score: {hdfc_cost_efficiency_score:.1f}/100")

HDFC Bank Cost-to-Income Ratio: 40.2%
HDFC Bank cost efficiency score: 59.8/100


## Jupiter — Burn multiple & EBITDA margin

Jupiter has no Cost-to-Income Ratio (it isn't a deposit-taking bank). The comparable
efficiency signals disclosed via its FY24 RoC filing (as reported by Entrackr) are:
- **Burn multiple** (net cash burned per unit of net new revenue): 6.45 — unverified/approximate, publication-derived
- **EBITDA margin**: −202.4%

We use EBITDA margin for the bounded score since it's on the same "percent of revenue"
footing as HDFC's CIR (both express operating cost burden relative to income).

In [3]:
jupiter_entity = "Jupiter (Amica Financial Technologies)"
jupiter_burn_multiple = df.loc[
    (df["entity"] == jupiter_entity) & (df["metric"] == "Burn multiple (implied)"), "value"
].iloc[0]
jupiter_ebitda_margin = df.loc[
    (df["entity"] == jupiter_entity) & (df["metric"] == "EBITDA margin"), "value"
].iloc[0]
print(f"Jupiter burn multiple: {jupiter_burn_multiple}x (unverified/approximate)")
print(f"Jupiter EBITDA margin: {jupiter_ebitda_margin}%")

# Bounded score: breakeven (0% margin) or better -> 100, -100pp of margin -> 0.
jupiter_cost_efficiency_score = max(0.0, 100.0 + jupiter_ebitda_margin)
print(f"Jupiter cost efficiency score: {jupiter_cost_efficiency_score:.1f}/100")

Jupiter burn multiple: 6.45x (unverified/approximate)
Jupiter EBITDA margin: -202.4%
Jupiter cost efficiency score: 0.0/100


In [4]:
scores = pd.DataFrame([
    {
        "entity": "HDFC Bank",
        "primary_metric": "Cost-to-Income Ratio",
        "primary_value": hdfc_cir,
        "primary_unit": "%",
        "confidence": "official",
        "cost_efficiency_score": round(hdfc_cost_efficiency_score, 1),
    },
    {
        "entity": "Jupiter",
        "primary_metric": "EBITDA margin (burn multiple: {:.2f}x)".format(jupiter_burn_multiple),
        "primary_value": jupiter_ebitda_margin,
        "primary_unit": "%",
        "confidence": "unverified/approximate",
        "cost_efficiency_score": round(jupiter_cost_efficiency_score, 1),
    },
])
scores.to_csv(PROCESSED / "cost_efficiency_scores.csv", index=False)
print("-> data/processed/cost_efficiency_scores.csv")
scores

-> data/processed/cost_efficiency_scores.csv


,entity,primary_metric,primary_value,primary_unit,confidence,cost_efficiency_score
0,HDFC Bank,Cost-to-Income Ratio,40.2,%,official,59.8
1,Jupiter,EBITDA margin (burn multiple: 6.45x),-202.4,%,unverified/approximate,0.0
